In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, get_peft_model_state_dict, PrefixTuningConfig, TaskType
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = "cuda"
model_name_or_path = "t5-large"
tokenizer_name_or_path = "t5-large"

text_column = "sentence"
label_column = "text_label"
max_length = 128
lr = 1e-2
num_epochs = 5
batch_size = 8

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "gtfintechlab/financial_phrasebank_sentences_allagree",
    "5768"
)

from datasets import ClassLabel

# Make sure every split's `label` is ClassLabel, regardless of current dtype
def ensure_classlabel(ds, column="label", names=("negative","neutral","positive")):
    for split in ds.keys():
        feat = ds[split].features[column]
        # If it's already ClassLabel, skip
        if isinstance(feat, ClassLabel):
            continue
        # If it's strings: encode to ClassLabel (derives mapping from data)
        if getattr(feat, "dtype", None) == "string":
            ds[split] = ds[split].class_encode_column(column)
        else:
            # If it's int Value, just attach the names (assumes 0/1/2 order)
            ds[split] = ds[split].cast_column(column, ClassLabel(names=list(names)))
    return ds

dataset = ensure_classlabel(dataset, "label", names=("negative","neutral","positive"))

# Now this works exactly like the tutorial:
classes = dataset["train"].features["label"].names

# Build text_label from integer ids (unchanged tutorial logic)
dataset = dataset.map(
    lambda x: {"text_label": [classes[i] for i in x["label"]]},
    batched=True,
    num_proc=1,
)

#dataset = load_dataset("lmassaron/FinancialPhraseBank", "sentences_allagree")
dataset = dataset["train"].train_test_split(test_size=0.1)
dataset["validation"] = dataset["test"]
del dataset["test"]

# classes = dataset["train"].features["label"].names
# dataset = dataset.map(
#     lambda x: {"text_label": [classes[label] for label in x["label"]]},
#     batched=True,
#     num_proc=1,
# )

dataset["train"][0]
{"sentence": "Profit before taxes was EUR 4.0 mn , down from EUR 4.9 mn .", "label": 0, "text_label": "negative"}

{'sentence': 'Profit before taxes was EUR 4.0 mn , down from EUR 4.9 mn .',
 'label': 0,
 'text_label': 'negative'}

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)


def preprocess_function(examples):
    inputs = examples[text_column]
    targets = examples[label_column]
    model_inputs = tokenizer(inputs, max_length=max_length, padding="max_length", truncation=True, return_tensors="pt")
    labels = tokenizer(targets, max_length=2, padding="max_length", truncation=True, return_tensors="pt")
    labels = labels["input_ids"]
    labels[labels == tokenizer.pad_token_id] = -100
    model_inputs["labels"] = labels
    return model_inputs

processed_datasets = dataset.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset["train"].column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

train_dataset = processed_datasets["train"]
eval_dataset = processed_datasets["validation"]

train_dataloader = DataLoader(
    train_dataset, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
eval_dataloader = DataLoader(eval_dataset, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)

Running tokenizer on dataset (num_proc=1):   0%|          | 0/1425 [00:00<?, ? examples/s]

Running tokenizer on dataset (num_proc=1):   0%|          | 0/159 [00:00<?, ? examples/s]

In [7]:
peft_config = PrefixTuningConfig(task_type=TaskType.SEQ_2_SEQ_LM, inference_mode=False, num_virtual_tokens=20)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

trainable params: 983,040 || all params: 738,651,136 || trainable%: 0.1331


100%|██████████| 20/20 [00:01<00:00, 15.55it/s]


epoch=0: train_ppl=tensor(9.0397, device='cuda:0') train_epoch_loss=tensor(2.2016, device='cuda:0') eval_ppl=tensor(1.1167, device='cuda:0') eval_epoch_loss=tensor(0.1104, device='cuda:0')


100%|██████████| 20/20 [00:01<00:00, 15.51it/s]


epoch=1: train_ppl=tensor(1.1826, device='cuda:0') train_epoch_loss=tensor(0.1677, device='cuda:0') eval_ppl=tensor(1.0706, device='cuda:0') eval_epoch_loss=tensor(0.0682, device='cuda:0')


...

epoch=2: train_ppl=tensor(1.1035, device='cuda:0') train_epoch_loss=tensor(0.0985, device='cuda:0') eval_ppl=tensor(1.0473, device='cuda:0') eval_epoch_loss=tensor(0.0463, device='cuda:0')


100%|██████████| 20/20 [00:01<00:00, 15.40it/s]


epoch=3: train_ppl=tensor(1.0970, device='cuda:0') train_epoch_loss=tensor(0.0926, device='cuda:0') eval_ppl=tensor(1.0433, device='cuda:0') eval_epoch_loss=tensor(0.0424, device='cuda:0')


100%|██████████| 20/20 [00:01<00:00, 15.34it/s]

epoch=4: train_ppl=tensor(1.0841, device='cuda:0') train_epoch_loss=tensor(0.0808, device='cuda:0') eval_ppl=tensor(1.0454, device='cuda:0') eval_epoch_loss=tensor(0.0444, device='cuda:0')


In [8]:
"""
model = AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path).to(device)

model.eval()
eval_loss = 0
eval_preds = []
for step, batch in enumerate(tqdm(eval_dataloader)):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    loss = outputs.loss
    eval_loss += loss.detach().float()
    eval_preds.extend(
        tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
    )
"""

correct = 0
total = 0
for pred, true in zip(eval_preds, dataset["validation"]["text_label"]):
    if pred.strip() == true.strip():
        correct += 1
    total += 1
accuracy = correct / total * 100
print(f"{accuracy=} % on the evaluation dataset")
print(f"{eval_preds[:10]=}")
print(f"{dataset['validation']['text_label'][:10]=}")

accuracy=98.11320754716981 % on the evaluation dataset
eval_preds[:10]=['positive', 'neutral', 'neutral', 'neutral', 'neutral', 'negative', 'neutral', 'negative', 'neutral', 'positive']
dataset['validation']['text_label'][:10]=['positive', 'neutral', 'neutral', 'neutral', 'neutral', 'negative', 'neutral', 'negative', 'neutral', 'positive']
